# Try local open-weight models with Ollama

This guide extends `tutorial_1_concept_analysis.ipynb`: replace the hosted LLM with an Ollama-served model, generate biological concepts from saved topic genes, and explore cell annotation, coherence, diversity, and UMAP. It is a model experimentation guide, not a manuscript reproduction workflow. Manuscript reproduction notebooks are in `Tutorial2/`.

First install scConcept and prepare the processed dataset and trained topic genes using the quickstart and Tutorial 1. This notebook does not train a topic model. The example uses Pollen, 50 topics, seed 1, epoch 500; change the dataset and topic-file settings to use your own outputs.

Requirements: an environment with the scConcept analysis dependencies (including PyTorch, SciPy, scikit-learn, Scanpy, AnnData, Matplotlib, and gseapy), an installed Ollama runtime, and sufficient RAM/VRAM for your chosen model and context. Larger contexts consume more memory and may offload to CPU. Run from within the cloned repository.


## 1. Install Ollama and obtain a model

On the machine running the notebook kernel, install Ollama using its [installation instructions](https://docs.ollama.com/). This notebook's background startup code targets Linux/macOS; if you use a managed Ollama service or Windows app, start that first and let the notebook connect to it.

For a manual Linux server, use a terminal on the kernel machine:

```bash
# Optional: put model weights on a disk with enough free space BEFORE starting the server.
# export OLLAMA_MODELS=/your/large_disk/ollama_models
export OLLAMA_HOST=127.0.0.1:11434
export OLLAMA_NO_CLOUD=1
ollama serve
```

If a server is already running, reuse it. In another terminal on the same machine, download the model you want to test (this requires network access and disk space):

```bash
export OLLAMA_HOST=127.0.0.1:11434
ollama pull qwen3:8b
ollama list
```

To try another locally served model, replace `qwen3:8b` in both `ollama pull` and the notebook's `MODEL` setting with its Ollama model tag. The notebook does not download weights. Use a local model, rather than a cloud model tag. Existing server processes determine their own cache and GPU settings; notebook settings affect only servers started by the notebook.

If you prefer notebook-managed startup, run the configuration cell after installing Ollama. It starts a detached server when necessary; if the chosen model is missing, download it in a terminal as above, then rerun that cell. Server logs and PID files are saved under `Results/ollama_tutorial/`. Detached startup does not provide automatic recovery after a host reboot; rerun the configuration cell to check/start the server.

`127.0.0.1` refers to the kernel machine, not the browser machine. Set `OLLAMA_BASE_URL` to your server address when needed; use the same host/port for the CLI's `OLLAMA_HOST`. For a suspended process (e.g. after Ctrl+Z), resume it with `fg` in its original terminal. The notebook avoids starting another server when an existing listener is unresponsive.


In [ ]:
import numpy as np
from types import SimpleNamespace
from pathlib import Path
import sys

# =========================================================
# Project setup for the tutorial notebook
# =========================================================

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "scConcept.py").exists() and (p / "ECRTM").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the scConcept repository.")
ECRTM_PATH = PROJECT_ROOT / "ECRTM"
sys.path.append(str(ECRTM_PATH))

from singlecell_dataset import SingleCellDataset

# =========================================================
# User-defined configuration
# =========================================================

dataset_name = "pollen"
data_dir = PROJECT_ROOT / "Datasets"
batch_size = 512

# =========================================================
# Build configuration namespace
# =========================================================

config = SimpleNamespace(
    dataset=dataset_name,
    batch_size=batch_size,
    data_dir=str(data_dir),
)

# =========================================================
# Load dataset
# =========================================================

dataset = SingleCellDataset(
    dataset_name=config.dataset,
    batch_size=config.batch_size,
    data_dir=config.data_dir,
)

train_matrix = dataset.train_data
test_matrix = dataset.test_data
train_labels = dataset.train_labels
test_labels = dataset.test_labels
gene_names = dataset.gene_names
num_genes = dataset.n_genes

# =========================================================
# Sanity check
# =========================================================

print(f"Train data shape: {train_matrix.shape}")
print(f"Test data shape: {test_matrix.shape}")
print(f"Train labels shape: {np.shape(train_labels)}")
print(f"Test labels shape: {np.shape(test_labels)}")
print(f"Number of genes: {num_genes}")
print(f"First 10 genes: {gene_names[:10]}")

In [ ]:
from pathlib import Path

# =========================================================
# Function: load topic gene lists
# =========================================================

def load_topic_genes(topic_file):
    """
    Load topic gene lists from a text file.

    Each line corresponds to one topic, where genes are separated by spaces.

    Parameters
    ----------
    topic_file : str or Path
        Path to the topic gene file.

    Returns
    -------
    list[list[str]]
        A list of topics, where each topic is a list of gene names.
    """
    topic_gene_lists = []

    with open(topic_file, "r", encoding="utf-8") as f:
        for line in f:
            genes = line.strip().split()
            topic_gene_lists.append(genes)

    return topic_gene_lists


# =========================================================
# Automatically construct topic file path
# =========================================================

num_topics = 50
seed = 1
epoch = 500

topic_dir = (
    PROJECT_ROOT
    / "ECRTM"
    / "output"
    / "topics"
    / f"{dataset_name}_K{num_topics}_seed{seed}"
)

topic_file = topic_dir / f"epoch{epoch}_top_genes.txt"

# Check file existence
if not topic_file.exists():
    raise FileNotFoundError(f"Topic file not found: {topic_file}")

# =========================================================
# Load topic gene lists
# =========================================================

topic_gene_lists = load_topic_genes(topic_file)

# =========================================================
# Sanity check
# =========================================================

print(f"Topic file: {topic_file}")
print(f"Number of topics: {len(topic_gene_lists)}")
print(f"Top 20 genes in Topic 0: {topic_gene_lists[0][:20]}")

## 2. Configure and generate concepts

Change `MODEL` to the exact installed Ollama tag. Set `NUM_CTX` to a context size supported by your model and available memory. This example requests 65,536 tokens for the full 50-topic input plus output; smaller contexts may truncate inputs. Inspect the server log for `truncating input prompt` and retain `prompt_eval_count` when assessing runs. Do not interpret a truncated-input run as a full-input test.

`STRUCTURED_OUTPUT=False` requests JSON; `True` adds schema constraints for the output shape, gene count, and input vocabulary. Some models/runtimes may not support all options. Qwen3 tags receive `think=False` and non-thinking sampling settings; other models use temperature 0. Adjust the request-building function if your model requires different controls. The output budget is 12,288 tokens (`num_predict` in `generate`); increase it, with sufficient context and memory, if `done_reason` is `length`.

The biological prompt retains all input topics and genes in their original order. A fixed seed does not ensure identical results across runtimes or hardware. No model configuration in this guide is claimed to guarantee valid or biologically accurate output.

Run configuration and function-definition cells first. Only the generation cell calls `/api/chat`. Every new run saves the complete request, raw response, parsed concepts when usable, and validation report. Set `REUSE_RUN_DIR` to a prior run to explore its saved response without generating again. Rerun cells from the top after restarting the kernel.

With `STRICT_VALIDATION=False`, usable concepts proceed despite gene validation failures. Cell scoring uses up to the first 100 returned genes and matches them to the expression vocabulary; unmatched genes do not contribute. This can substantially distort metrics. Strict mode blocks failed responses; it does not repair them.


In [ ]:
import os
import json
import shutil
import subprocess
import time
import urllib.request
import urllib.error
from datetime import datetime, timezone
from urllib.parse import urlparse

MODEL = "qwen3:8b"  # Or another model already installed in your Ollama server.
OLLAMA_BASE_URL = os.environ.get(
    "OLLAMA_BASE_URL", "http://127.0.0.1:11434"
).rstrip("/")
AUTO_START_SERVER = True
GPU_DEVICES = os.environ.get("SCCONCEPT_OLLAMA_GPU")  # Optional, e.g. "0"; respect existing visibility otherwise.
NUM_CTX = 65536
STRICT_VALIDATION = False  # Exploratory mode: report gene validation failures but continue.
STRUCTURED_OUTPUT = False  # Set True to constrain JSON structure and input vocabulary.

# New runs use timestamped directories; existing results are retained.
WORK_DIR = PROJECT_ROOT / "Results/ollama_tutorial"
WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT = WORK_DIR / "results"
DIRECT_HTTP = urllib.request.build_opener(urllib.request.ProxyHandler({}))

def server_get(endpoint, timeout=5):
    with DIRECT_HTTP.open(OLLAMA_BASE_URL + endpoint, timeout=timeout) as response:
        return json.load(response)

server_log = None
try:
    version_info = server_get("/api/version")
except urllib.error.HTTPError as exc:
    raise RuntimeError(f"The server at {OLLAMA_BASE_URL} returned HTTP {exc.code}; check its address.") from exc
except (urllib.error.URLError, TimeoutError, OSError) as exc:
    # A read timeout may mean a paused/stuck listener; do not start a duplicate.
    if isinstance(exc, TimeoutError) or isinstance(getattr(exc, "reason", None), TimeoutError):
        raise RuntimeError(
            f"{OLLAMA_BASE_URL} accepts connections but did not respond. "
            "Check the Ollama terminal/process: Ctrl+Z suspends it; use fg in that terminal to resume. "
            "A second server was not started."
        ) from exc
    if not AUTO_START_SERVER:
        raise RuntimeError(f"Start Ollama at {OLLAMA_BASE_URL}, then rerun this cell.") from exc
    address = urlparse(OLLAMA_BASE_URL)
    if address.scheme != "http" or address.hostname not in ("127.0.0.1", "localhost"):
        raise RuntimeError("Automatic startup supports a local HTTP server only. Start the configured server manually.") from exc
    executable = shutil.which("ollama")
    if not executable:
        raise RuntimeError("Ollama is not installed. Install it and download your chosen model separately, then rerun.") from exc
    env = os.environ.copy()
    env.update(OLLAMA_HOST=f"127.0.0.1:{address.port or 80}", OLLAMA_NO_CLOUD="1", OLLAMA_NUM_PARALLEL="1")
    if GPU_DEVICES is not None:
        env.update(CUDA_VISIBLE_DEVICES=GPU_DEVICES, GGML_VK_VISIBLE_DEVICES=GPU_DEVICES)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    server_log = WORK_DIR / f"ollama_server_{stamp}.log"
    with server_log.open("xb") as log:
        process = subprocess.Popen([executable, "serve"], stdin=subprocess.DEVNULL,
                                   stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True)
    (WORK_DIR / f"ollama_server_{stamp}.pid").write_text(str(process.pid))
    deadline = time.monotonic() + 90
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError(f"Ollama exited with code {process.returncode}. Inspect {server_log}")
        try:
            version_info = server_get("/api/version", timeout=2)
            break
        except (urllib.error.URLError, TimeoutError, OSError):
            time.sleep(1)
    else:
        raise RuntimeError(f"Ollama was not ready within 90 seconds. Inspect {server_log}; no generation was sent.")
    print(f"Started background server PID {process.pid}; log: {server_log}")

installed_models = [m["name"] for m in server_get("/api/tags", timeout=15).get("models", [])]
model_name = MODEL if ":" in MODEL.rsplit("/", 1)[-1] else MODEL + ":latest"
if model_name not in installed_models:
    raise RuntimeError(
        f"Model {MODEL!r} is not installed at {OLLAMA_BASE_URL}. Available: {installed_models}. "
        "Select an installed model or run ollama pull MODEL against this server (see instructions above). Nothing was downloaded."
    )
MODEL = model_name
print(f"Ready: {OLLAMA_BASE_URL}; Ollama {version_info.get('version')}; model {MODEL}")
print("No inference has been sent. Continue to the generation cell when ready.")

# Optional: set a previous run directory to analyze its saved response without inference.
REUSE_RUN_DIR = None


In [ ]:
from datetime import datetime, timezone

import hashlib

import json

from pathlib import Path

import time

import urllib.request

def post(base_url, endpoint, payload, timeout=7200):
    request = urllib.request.Request(
        base_url.rstrip('/') + endpoint,
        data=json.dumps(payload).encode(),
        headers={'Content-Type': 'application/json'},
    )
    with DIRECT_HTTP.open(request, timeout=timeout) as response:
        return json.load(response)

def validate_concepts(result, topics):
    errors = []
    concepts = result.get('concepts') if isinstance(result, dict) else None
    if not isinstance(concepts, list) or not concepts:
        return ['Expected a nonempty concepts list.']
    by_id = {f'topic_{i}': set(genes) for i, genes in enumerate(topics)}
    for i, concept in enumerate(concepts):
        if not isinstance(concept, dict):
            errors.append(f'Concept {i}: expected an object.')
            continue
        for key in ('name', 'description'):
            if not isinstance(concept.get(key), str) or not concept[key].strip():
                errors.append(f'Concept {i}: missing {key}.')
        sources = concept.get('source_topics')
        if not isinstance(sources, list) or not sources or any(not isinstance(s, str) or s not in by_id for s in sources):
            errors.append(f'Concept {i}: invalid source_topics.')
            continue
        genes = concept.get('genes')
        if not isinstance(genes, list) or any(not isinstance(g, str) for g in genes):
            errors.append(f'Concept {i}: invalid genes.')
            continue
        if len(genes) != 100 or len(set(genes)) != 100:
            errors.append(f'Concept {i}: expected 100 unique genes, got {len(genes)} entries and {len(set(genes))} unique.')
        outside = sorted(set(genes) - set().union(*(by_id[s] for s in sources)))
        if outside:
            errors.append(f'Concept {i}: genes outside declared source topics: {outside}.')
    return errors

def concept_schema(topics):
    genes = sorted({gene for topic in topics for gene in topic})
    return {
        "type": "object", "required": ["concepts"], "additionalProperties": False,
        "properties": {"concepts": {
            "type": "array", "minItems": 1, "maxItems": len(topics),
            "items": {
                "type": "object", "required": ["name", "description", "genes", "source_topics"],
                "additionalProperties": False,
                "properties": {
                    "name": {"type": "string"}, "description": {"type": "string"},
                    "genes": {"type": "array", "minItems": 100, "maxItems": 100,
                              "items": {"type": "string", "enum": genes}},
                    "source_topics": {"type": "array", "minItems": 1, "maxItems": len(topics),
                                      "items": {"type": "string", "enum": [f"topic_{i}" for i in range(len(topics))]}},
                },
            },
        }},
    }

def generate(model, topic_file, output_dir, base_url=None, num_ctx=65536, structured=False):
    base_url = base_url or OLLAMA_BASE_URL
    topic_file = Path(topic_file)
    topics = [line.split() for line in topic_file.read_text().splitlines() if line.strip()]
    if not topics:
        raise ValueError('No topics found.')
    output_dir = Path(output_dir)
    show = post(base_url, '/api/show', {'model': model}, timeout=30)
    output_dir.mkdir(parents=True, exist_ok=False)
    (output_dir / 'model.json').write_text(json.dumps(show, indent=2))
    runtime = {}
    for endpoint in ('version', 'tags'):
        with DIRECT_HTTP.open(base_url.rstrip('/') + '/api/' + endpoint, timeout=15) as response:
            runtime[endpoint] = json.load(response)
    (output_dir / 'runtime.json').write_text(json.dumps(runtime, indent=2))
    request = {
        'model': model,
        'messages': [
            {'role': 'system', 'content': 'You are an expert in single-cell biology.'},
            {'role': 'user', 'content': build_prompt(topics)},
        ],
        'stream': False,
        'format': concept_schema(topics) if structured else 'json',
        'options': {'seed': 1, 'temperature': 0, 'num_ctx': num_ctx, 'num_predict': 12288},
        'keep_alive': 0,
    }
    if model.startswith('qwen3:'):
        request['think'] = False
        request['options'].update(temperature=0.7, top_p=0.8, top_k=20, min_p=0)
    (output_dir / 'request.json').write_text(json.dumps(request, indent=2))
    started = time.monotonic()
    print(f"Generating with {model}; full request saved in {output_dir}. This can take several minutes.", flush=True)
    try:
        response = post(base_url, '/api/chat', request)
    except (urllib.error.URLError, TimeoutError, OSError) as exc:
        (output_dir / 'transport_error.json').write_text(json.dumps({'error': str(exc), 'base_url': base_url}, indent=2))
        raise RuntimeError(f"Generation request failed; inspect {output_dir} and the Ollama server log. No automatic retry was sent.") from exc
    (output_dir / 'response.json').write_text(json.dumps(response, indent=2))
    raw = response.get('message', {}).get('content', '')
    (output_dir / 'raw_response.txt').write_text(raw)
    errors = []
    if response.get('done_reason') != 'stop':
        errors.append(f"Generation did not finish normally: {response.get('done_reason')}")
    try:
        result = json.loads(raw)
        errors.extend(validate_concepts(result, topics))
    except (ValueError, TypeError) as exc:
        result = None
        errors.append(f'Invalid JSON: {exc}')
    report = {
        'model': model, 'utc': datetime.now(timezone.utc).isoformat(),
        'topic_file': str(topic_file), 'topic_sha256': hashlib.sha256(topic_file.read_bytes()).hexdigest(),
        'topic_count': len(topics), 'elapsed_seconds': time.monotonic() - started,
        'prompt_eval_count': response.get('prompt_eval_count'), 'eval_count': response.get('eval_count'),
        'valid': not errors, 'errors': errors,
        'concept_count': len(result.get('concepts', [])) if isinstance(result, dict) else 0,
    }
    (output_dir / 'validation.json').write_text(json.dumps(report, indent=2))
    # Preserve the model's parsed output unchanged, including gene validation failures.
    usable = (
        isinstance(result, dict) and isinstance(result.get('concepts'), list)
        and bool(result['concepts'])
        and all(isinstance(c, dict) and isinstance(c.get('name'), str)
                and bool(c['name'].strip()) and isinstance(c.get('genes'), list)
                and bool(c['genes']) and all(isinstance(g, str) for g in c['genes'])
                for c in result['concepts'])
    )
    report['usable_for_exploration'] = usable
    report['strict_validation'] = STRICT_VALIDATION
    (output_dir / 'validation.json').write_text(json.dumps(report, indent=2))
    if usable:
        (output_dir / 'concepts.json').write_text(json.dumps(result['concepts'], indent=2))
    print(json.dumps(report, indent=2), flush=True)
    return report

import json

TOPIC_DISTILLATION_TEMPLATE = """
You are given topics derived from a neural topic model on single-cell RNA-seq.

Each topic includes ONLY:
- A list of the top 100 genes for that topic,
  sorted from highest weight to lowest weight in the model.

Here are all topics (JSON list):
__TOPICS_JSON__

TASK:
You must perform biological topic distillation by analyzing ONLY the top-100 gene lists.

Specifically:
1. Identify topics whose top genes indicate highly similar biological pathways or cell states.
2. Merge such topics into unified, biologically coherent concepts.
3. Remove topics that are:
   - too vague (gene list does not form a coherent module),
   - too narrow (dominated by a single outlier gene),
   - biologically uninterpretable given the provided gene patterns.

For each FINAL biological concept, output:
- "name": a concise 2-4 word biological label.
      * Avoid generic labels like "General Regulation".
      * Prefer pathway- or state-specific terms.
- "description": a 10-30 word natural language description
      * Summarize the biological meaning of this concept.
      * Describe the core pathway, cellular program, or functional state.
      * Must be consistent with the concept name and gene list.
      * Do NOT mention topic IDs, gene counts, or modeling details.
- "genes": EXACTLY 100 representative genes.
      * MUST be chosen from the union of gene lists of the merged source topics.
      * MUST reflect a coherent pathway or co-expression program.
      * MUST respect the original sorted importance: higher-ranked genes are preferred.
      * Do NOT invent gene names.
- "source_topics": list of topic IDs merged into this concept.

OUTPUT FORMAT (STRICT):
Respond ONLY with a valid JSON object in EXACTLY this form:

{
  "concepts": [
    {
      "name": "Example Name",
      "description": "A concise biological description summarizing the core pathway or cellular program represented by this concept.",
      "genes": ["GENE1", "GENE2", "GENE3", "...", "GENE100"],
      "source_topics": ["topic_0", "topic_3"]
    }
  ]
}

ABSOLUTE RULES:
- DO NOT output anything outside the JSON.
- DO NOT add code fences.
- DO NOT invent genes.
- The final genes must reflect the most informative subset of the top 100 genes per topic.
"""

def build_prompt(topic_gene_lists, compact=True):
    topics_payload = [{"topic_id": f"topic_{i}", "genes": genes} for i, genes in enumerate(topic_gene_lists)]
    topics_json_str = json.dumps({"topics": topics_payload}, separators=(",", ":")) if compact else json.dumps({"topics": topics_payload}, indent=2)
    return TOPIC_DISTILLATION_TEMPLATE.replace("__TOPICS_JSON__", topics_json_str)



In [ ]:
# Reuse a saved response without calling the model, or generate a new response.
if REUSE_RUN_DIR is not None:
    results_dir = Path(REUSE_RUN_DIR)
    report = json.loads((results_dir / "validation.json").read_text())
    result = json.loads((results_dir / "raw_response.txt").read_text())
    concepts = result.get("concepts") if isinstance(result, dict) else None
    if not isinstance(concepts, list) or not concepts or not all(
        isinstance(c, dict) and isinstance(c.get("name"), str) and bool(c["name"].strip())
        and isinstance(c.get("genes"), list) and bool(c["genes"])
        and all(isinstance(g, str) for g in c["genes"]) for c in concepts
    ):
        raise ValueError("Saved response lacks the concept fields required for analysis.")
    if STRICT_VALIDATION and not report["valid"]:
        raise ValueError("Saved response failed strict validation.")
    # Keep all previous result files intact; save analysis in a new sibling folder.
    source_run_dir = results_dir
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    results_dir = source_run_dir.parent / f"{source_run_dir.name}_exploratory_{stamp}"
    results_dir.mkdir(parents=True, exist_ok=False)
    (results_dir / "concepts.json").write_text(json.dumps(concepts, indent=2))
    (results_dir / "validation.json").write_text(json.dumps(report, indent=2))
    (results_dir / "source_run.json").write_text(json.dumps({"source_run": str(source_run_dir)}, indent=2))
    MODEL = report["model"]
    print(f"Loaded {len(concepts)} saved concepts without inference. Validation passed: {report['valid']}")
    print(f"Exploratory analysis outputs: {results_dir}")
else:
    # Run generation: this cell calls the local model.
    # Avoid retaining concepts from an earlier successful run if this attempt fails.
    concepts = None
    try:
        server_get("/api/version", timeout=5)
    except (urllib.error.URLError, TimeoutError, OSError) as exc:
        raise RuntimeError("Ollama is unavailable. Rerun the configuration/server cell before generating.") from exc
    run_name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    model_tag = MODEL.replace(":", "_").replace("/", "_")
    results_dir = OUTPUT_ROOT / dataset_name / "ollama" / model_tag / run_name
    report = generate(MODEL, topic_file, results_dir, OLLAMA_BASE_URL, NUM_CTX, structured=STRUCTURED_OUTPUT)
    if not report["usable_for_exploration"]:
        raise ValueError(f"The response cannot be used for analysis (invalid JSON or missing name/genes). Inspect {results_dir}.")
    if not report["valid"]:
        if STRICT_VALIDATION:
            raise ValueError(f"Validation failed. Inspect {results_dir}; set STRICT_VALIDATION=False for exploratory analysis.")
        import warnings
        warnings.warn(
            f"Exploratory run: {len(report['errors'])} validation issues retained. "
            f"Genes were not repaired or removed. See {results_dir / 'validation.json'}. "
            "Downstream metrics describe the unvalidated model output."
        )
    concepts = json.loads((results_dir / "concepts.json").read_text())
    print(f"Loaded {len(concepts)} concepts from {results_dir}")


In [ ]:
if not concepts:
    raise RuntimeError("Load a usable concept set before downstream analysis; inspect validation.json on failure.")

import numpy as np
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score

# =========================================================
# Utility functions for clustering evaluation
# =========================================================

def compute_clustering_metrics(true_labels, score_matrix):
    """
    Compute purity, adjusted mutual information (AMI), and adjusted Rand index (ARI)
    based on the highest-scoring concept assigned to each cell.

    Parameters
    ----------
    true_labels : array-like
        Ground-truth labels for cells.
    score_matrix : np.ndarray
        Matrix of shape (num_cells, num_concepts), where each row contains
        concept scores for one cell.

    Returns
    -------
    purity : float
        Clustering purity.
    ami : float
        Adjusted mutual information.
    ari : float
        Adjusted Rand index.
    """
    true_clusters = {}
    predicted_clusters = {}
    num_cells = len(true_labels)

    predicted_indices = np.argmax(score_matrix, axis=1)

    # Group cells by ground-truth labels
    for cell_index, label in enumerate(true_labels):
        true_clusters.setdefault(label, set()).add(cell_index)

    # Group cells by predicted concept assignment
    for cell_index, concept_index in enumerate(predicted_indices):
        cluster_name = f"Concept_{concept_index}"
        predicted_clusters.setdefault(cluster_name, set()).add(cell_index)

    # Compute purity
    correct_count = 0
    for predicted_cells in predicted_clusters.values():
        best_overlap = 0
        for true_cells in true_clusters.values():
            best_overlap = max(best_overlap, len(predicted_cells.intersection(true_cells)))
        correct_count += best_overlap

    purity = correct_count / num_cells
    ami = adjusted_mutual_info_score(true_labels, predicted_indices)
    ari = adjusted_rand_score(true_labels, predicted_indices)

    return purity, ami, ari


def flatten_labels(labels):
    """
    Convert labels into a one-dimensional NumPy array.

    This is useful when labels are stored as nested arrays or object arrays.

    Parameters
    ----------
    labels : array-like
        Input labels.

    Returns
    -------
    np.ndarray
        Flattened label array.
    """
    labels = np.array(labels, dtype=object).reshape(-1)
    flattened = []

    for label in labels:
        if isinstance(label, np.ndarray):
            if label.size == 1:
                flattened.append(label.item())
            else:
                flattened.append(tuple(label.tolist()))
        else:
            flattened.append(label)

    return np.array(flattened)


# =========================================================
# Utility functions for concept-based cell annotation
# =========================================================

def build_gene_index(gene_names):
    """
    Create a mapping from gene name to column index.

    Parameters
    ----------
    gene_names : list-like
        Ordered list of gene names.

    Returns
    -------
    dict
        Dictionary mapping gene name to column index.
    """
    return {str(gene): idx for idx, gene in enumerate(gene_names)}


def truncate_concept_genes(concepts, top_k):
    """
    Keep only the top-k genes for each concept.

    Parameters
    ----------
    concepts : list[dict]
        List of concept dictionaries. Each concept should contain at least:
        - "name"
        - "genes"
        - optional "source_topics"
    top_k : int
        Number of top genes to retain.

    Returns
    -------
    list[dict]
        Updated concept list with truncated gene sets.
    """
    truncated_concepts = []

    for concept in concepts:
        truncated_concepts.append(
            {
                "name": concept["name"],
                "genes": concept["genes"][:top_k],
                "source_topics": concept.get("source_topics", [])
            }
        )

    return truncated_concepts


def assign_cells_to_concepts_by_zscore(concepts, expression_matrix, gene_to_index):
    """
    Assign each cell to the concept with the highest average marker z-score.

    Parameters
    ----------
    concepts : list[dict]
        List of concept dictionaries. Each concept should contain:
        - "name"
        - "genes"
    expression_matrix : torch.Tensor
        Cell-by-gene expression matrix of shape (num_cells, num_genes).
    gene_to_index : dict
        Mapping from gene name to column index.

    Returns
    -------
    score_matrix : np.ndarray
        Matrix of shape (num_cells, num_concepts).
    concept_names : list[str]
        Names of all concepts.
    predicted_labels : list[str]
        Predicted concept label for each cell.
    """
    expression_array = expression_matrix.detach().cpu().numpy().astype(float)
    num_cells, _ = expression_array.shape

    # Standardize each gene across cells
    gene_mean = expression_array.mean(axis=0, keepdims=True)
    gene_std = expression_array.std(axis=0, keepdims=True) + 1e-6
    expression_zscore = (expression_array - gene_mean) / gene_std

    concept_names = [concept["name"] for concept in concepts]
    num_concepts = len(concepts)

    score_matrix = np.zeros((num_cells, num_concepts), dtype=float)

    for concept_index, concept in enumerate(concepts):
        concept_genes = concept["genes"]
        gene_indices = [
            gene_to_index[gene] for gene in concept_genes if gene in gene_to_index
        ]

        if len(gene_indices) == 0:
            continue

        score_matrix[:, concept_index] = expression_zscore[:, gene_indices].mean(axis=1)

    predicted_indices = np.argmax(score_matrix, axis=1)
    predicted_labels = [concept_names[idx] for idx in predicted_indices]

    return score_matrix, concept_names, predicted_labels


# =========================================================
# Example usage: concept-based annotation on the test set
# =========================================================

gene_to_index = build_gene_index(gene_names)

top_k_concepts = truncate_concept_genes(concepts, top_k=100)

score_matrix, concept_names, predicted_concepts = assign_cells_to_concepts_by_zscore(
    top_k_concepts,
    test_matrix,
    gene_to_index
)

print(f"Score matrix shape: {score_matrix.shape}")
print(f"Example predicted concepts: {predicted_concepts[:10]}")


# =========================================================
# Evaluate concept-based assignments
# =========================================================

true_test_labels = flatten_labels(test_labels)

print(f"Label example type: {type(true_test_labels[0])}")
print(f"First 10 ground-truth labels: {true_test_labels[:10]}")

purity, ami, ari = compute_clustering_metrics(true_test_labels, score_matrix)

print(f"Purity: {purity:.4f}")
print(f"ARI: {ari:.4f}")
print(f"AMI: {ami:.4f}")
from sklearn.metrics import normalized_mutual_info_score
nmi = normalized_mutual_info_score(true_test_labels, np.argmax(score_matrix, axis=1))
print(f"NMI: {nmi:.4f}")


In [ ]:
import numpy as np
from collections import Counter
from pathlib import Path
import gseapy as gp

# =========================================================
# Utility function: compute topic coherence (TC)
# =========================================================

def compute_topic_coherence_from_gene_lists(
    background_matrix,
    background_vocab,
    concept_gene_lists,
    gene_index_mapping,
    top_n=10
):
    """
    Compute topic coherence (TC) from concept gene lists.

    Parameters
    ----------
    background_matrix : np.ndarray
        Binary pathway-by-gene matrix of shape (num_pathways, num_background_genes).
    background_vocab : list[str]
        Background gene vocabulary corresponding to the columns of background_matrix.
    concept_gene_lists : list[list[str]]
        Concept gene lists, where each concept is represented by a list of gene names.
    gene_index_mapping : dict
        Mapping from vocabulary index to background_matrix column index.
    top_n : int, default=10
        Number of top genes used for coherence calculation.

    Returns
    -------
    float
        Average topic coherence score across all concepts.
    """
    num_concepts = len(concept_gene_lists)
    num_documents = background_matrix.shape[0]

    gene_to_vocab_index = {gene: idx for idx, gene in enumerate(background_vocab)}

    # Convert gene lists into index lists
    indexed_concepts = []
    for genes in concept_gene_lists:
        gene_indices = []

        for gene in genes:
            gene_indices.append(gene_to_vocab_index.get(gene, -1))

        if len(gene_indices) > top_n:
            gene_indices = gene_indices[:top_n]
        elif len(gene_indices) < top_n:
            gene_indices = gene_indices + [-1] * (top_n - len(gene_indices))

        indexed_concepts.append(np.array(gene_indices, dtype=int))

    total_coherence = 0.0

    for concept_indices in indexed_concepts:
        concept_score = 0.0

        for i in range(top_n):
            gene_i = concept_indices[i]

            if gene_i in gene_index_mapping:
                presence_i = background_matrix[:, gene_index_mapping[gene_i]] > 0
                prob_i = np.sum(presence_i) / num_documents

                for j in range(i + 1, top_n):
                    gene_j = concept_indices[j]

                    if gene_j in gene_index_mapping:
                        presence_j = background_matrix[:, gene_index_mapping[gene_j]] > 0
                        count_j = np.sum(presence_j)
                        count_ij = np.sum(presence_i * presence_j)

                        if prob_i * count_j * count_ij > 0:
                            prob_j = count_j / num_documents
                            prob_ij = count_ij / num_documents

                            if (prob_i * prob_j) == 1 and prob_ij == 1:
                                continue

                            concept_score += np.log(prob_ij / (prob_i * prob_j)) / -np.log(prob_ij)

        total_coherence += concept_score * (2 / (top_n * top_n - top_n))

    return total_coherence / num_concepts


# =========================================================
# Utility function: compute topic diversity (TD)
# =========================================================

def compute_topic_diversity(concept_gene_lists):
    """
    Compute topic diversity (TD), defined as the fraction of genes
    that appear exactly once across all concept gene lists.

    Parameters
    ----------
    concept_gene_lists : list[list[str]]
        List of concept gene lists.

    Returns
    -------
    float
        Topic diversity score.
    """
    num_concepts = len(concept_gene_lists)
    genes_per_concept = len(concept_gene_lists[0])

    all_genes = []
    for gene_list in concept_gene_lists:
        all_genes.extend(gene_list)

    gene_counts = Counter(all_genes)
    unique_gene_count = sum(1 for count in gene_counts.values() if count == 1)

    return unique_gene_count / (num_concepts * genes_per_concept)


# =========================================================
# Load pathway gene sets from GMT
# =========================================================
# This example uses the supplied MSigDB GMT collection as the background
# reference for concept coherence evaluation.

gmt_file = PROJECT_ROOT / "Datasets" / "msigdb.v2024.1.Hs.symbols.gmt"

if not gmt_file.exists():
    raise FileNotFoundError(f"GMT file not found: {gmt_file}")

kegg_pathways = gp.read_gmt(path=str(gmt_file))

# =========================================================
# Build the background vocabulary
# =========================================================

background_vocab = sorted(
    list({gene.upper() for genes in kegg_pathways.values() for gene in genes})
)
background_gene_to_index = {
    gene: idx for idx, gene in enumerate(background_vocab)
}

# Build a binary pathway-by-gene matrix
background_matrix = np.zeros(
    (len(kegg_pathways), len(background_vocab)),
    dtype=float
)

for pathway_index, genes in enumerate(kegg_pathways.values()):
    for gene in genes:
        background_matrix[pathway_index, background_gene_to_index[gene.upper()]] = 1.0

# =========================================================
# Prepare concept gene lists
# =========================================================
# Convert concept genes to uppercase and keep the top 10 genes
# for each concept when evaluating TC and TD.

concept_gene_lists_upper = [
    [str(gene).upper() for gene in concept.get("genes", [])[:10]]
    for concept in top_k_concepts
]

# =========================================================
# Build the vocabulary-to-column mapping
# =========================================================
# Here the background vocabulary and matrix columns are aligned,
# so the mapping is simply identity.

gene_index_mapping = {idx: idx for idx in range(len(background_vocab))}

# =========================================================
# Compute TC and TD
# =========================================================

topic_coherence = compute_topic_coherence_from_gene_lists(
    background_matrix=background_matrix,
    background_vocab=background_vocab,
    concept_gene_lists=concept_gene_lists_upper,
    gene_index_mapping=gene_index_mapping,
    top_n=10
)

topic_diversity = compute_topic_diversity(concept_gene_lists_upper)

print(f"TC: {topic_coherence:.4f}")
print(f"TD: {topic_diversity:.4f}")

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad

# =========================================================
# Prepare the expression matrix and labels
# =========================================================

expression_array = test_matrix.detach().cpu().numpy()
true_cell_types = flatten_labels(test_labels)
predicted_concepts_array = np.array(predicted_concepts)

print(f"Expression matrix shape: {expression_array.shape}")
print(f"Cell type labels shape: {true_cell_types.shape}")
print(f"Predicted concept labels shape: {predicted_concepts_array.shape}")

# =========================================================
# Build an AnnData object for visualization
# =========================================================

adata_visualization = ad.AnnData(expression_array)

adata_visualization.obs["cell_type"] = pd.Categorical(true_cell_types)
adata_visualization.obs["concept"] = pd.Categorical(predicted_concepts_array)

# Add gene names
adata_visualization.var_names = [str(gene) for gene in gene_names]

# =========================================================
# Compute PCA, neighborhood graph, and UMAP
# =========================================================

sc.pp.pca(adata_visualization, n_comps=50)
sc.pp.neighbors(adata_visualization, n_neighbors=15, n_pcs=50)
sc.tl.umap(adata_visualization, random_state=1)

# =========================================================
# Visualize UMAP colored by cell type and concept
# =========================================================

figure = sc.pl.umap(
    adata_visualization,
    color=["cell_type", "concept"],
    ncols=2,
    frameon=False,
    wspace=0.35,
    legend_loc="right margin",
    return_fig=True
)

# Optionally enlarge legend font size
for axis in figure.axes[:2]:
    if axis.legend_ is not None:
        for text in axis.legend_.get_texts():
            text.set_fontsize(12)

# =========================================================
# Save the figure
# =========================================================


figure_file = results_dir / "umap_celltype_concept.pdf"
figure.savefig(figure_file, bbox_inches="tight")

print(f"UMAP figure saved to: {figure_file}")

In [ ]:
metrics = {"model": MODEL, "dataset": dataset_name, "purity": float(purity),
           "ARI": float(ari), "AMI": float(ami), "NMI": float(nmi),
           "TC": float(topic_coherence), "TD": float(topic_diversity)}
metrics.update({"validation_passed": bool(report["valid"]),
                "strict_validation": STRICT_VALIDATION,
                "exploratory_unvalidated": not report["valid"],
                "validation_error_count": len(report["errors"]),
                "genes_found_in_expression": {
                    c["name"]: sum(g in gene_to_index for g in c["genes"][:100])
                    for c in concepts
                }})
(results_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))


## Output guide

| Analysis | Main output |
| --- | --- |
| Generation settings and prompt | `request.json` |
| Full model response | `response.json`, `raw_response.txt` |
| Model information | `model.json`, `runtime.json` |
| Output checks | `validation.json` |
| Validated concept set | `concepts.json` |
| Clustering and interpretability metrics | `metrics.json` |
| Cell-type and concept UMAPs | `umap_celltype_concept.pdf` |

Generation files are stored in `Results/ollama_tutorial/results/<dataset>/ollama/<model>/<run>/`. Repeat the notebook with the other model name to compare outputs using the same dataset and topic file.
